In [1]:
# ============================================================
# UPLOAD DE ÁUDIOS + INSERÇÃO ALEATÓRIA DE RUÍDOS
# 1 ruidoso por limpo, duração completa do áudio limpo
# ============================================================
# !pip install -q librosa soundfile

import os, glob, random, shutil
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm

# ---------------------------------------------------------------------
# CONFIGURAÇÕES
# ---------------------------------------------------------------------
SR = 16000

SNR_MIN_DB = -12.0
SNR_MAX_DB = 15.0

RMS_ALVO = 0.1
PICO_MAX = 0.95
RMS_MIN_LIMIAR = 0.005

EXTENSOES_AUDIO = ("*.wav", "*.WAV", "*.mp3", "*.flac", "*.ogg")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------------------------------------------------------------------
# FUNÇÕES AUXILIARES
# ---------------------------------------------------------------------
def listar_audios(pasta):
    arquivos = []
    for ext in EXTENSOES_AUDIO:
        arquivos.extend(glob.glob(os.path.join(pasta, "**", ext), recursive=True))
    return sorted(arquivos)

def carregar_audio(caminho, sr=SR):
    audio, _ = librosa.load(caminho, sr=sr, mono=True)
    return audio.astype(np.float32)

def rms(x):
    return float(np.sqrt(np.mean(x ** 2) + 1e-12))

def normalizar_rms(audio, rms_alvo=RMS_ALVO):
    r = rms(audio)
    return audio * (rms_alvo / (r + 1e-12))

def misturar_com_snr(limpo, ruido, snr_db):
    limpo_n = normalizar_rms(limpo)
    fator = (rms(limpo_n) / (rms(ruido) + 1e-12)) / (10 ** (snr_db / 20))
    ruidoso = limpo_n + ruido * fator
    pico = max(np.max(np.abs(limpo_n)), np.max(np.abs(ruidoso))) + 1e-9
    if pico > PICO_MAX:
        escala = PICO_MAX / pico
        limpo_n = limpo_n * escala
        ruidoso = ruidoso * escala
    return limpo_n.astype(np.float32), ruidoso.astype(np.float32)

def extrair_trecho_aleatorio(audio, comprimento):
    """
    Retorna um trecho de `comprimento` amostras retirado aleatoriamente do áudio.
    Se `audio` for menor que `comprimento`, faz tile (loop) até preencher.
    """
    if len(audio) < comprimento:
        reps = int(np.ceil(comprimento / len(audio)))
        audio = np.tile(audio, reps)
    inicio_max = len(audio) - comprimento
    inicio = random.randint(0, inicio_max) if inicio_max > 0 else 0
    return audio[inicio:inicio + comprimento]

# ---------------------------------------------------------------------
# 1. UPLOAD DOS ÁUDIOS (Colab) OU PASTA LOCAL
# ---------------------------------------------------------------------
def pedir_upload():
    try:
        from google.colab import files
        print("📤 Faça upload dos ÁUDIOS LIMPOS (motores, vozes, etc.):")
        up_limpos = files.upload()
        print("\n📤 Faça upload dos RUÍDOS (ambientes, vento, etc.):")
        up_ruidos = files.upload()

        os.makedirs("/content/audios_limpos", exist_ok=True)
        os.makedirs("/content/audios_ruidos", exist_ok=True)

        for nome in up_limpos:
            shutil.move(nome, os.path.join("/content/audios_limpos", nome))
        for nome in up_ruidos:
            shutil.move(nome, os.path.join("/content/audios_ruidos", nome))

        return "/content/audios_limpos", "/content/audios_ruidos"

    except ImportError:
        dir_limpos = input("Caminho da pasta com ÁUDIOS LIMPOS: ").strip()
        dir_ruidos = input("Caminho da pasta com RUÍDOS: ").strip()
        return dir_limpos, dir_ruidos

DIR_LIMPOS, DIR_RUIDOS = pedir_upload()
DIR_SAIDA = "/content/audios_misturados"
os.makedirs(DIR_SAIDA, exist_ok=True)

# ---------------------------------------------------------------------
# 2. LISTAR ARQUIVOS
# ---------------------------------------------------------------------
print("\nProcurando áudios...")
arquivos_limpos = listar_audios(DIR_LIMPOS)
arquivos_ruidos = listar_audios(DIR_RUIDOS)
print(f"  Áudios limpos: {len(arquivos_limpos)}")
print(f"  Ruídos       : {len(arquivos_ruidos)}")

assert arquivos_limpos, "❌ Nenhum áudio limpo encontrado."
assert arquivos_ruidos, "❌ Nenhum ruído encontrado."

# ---------------------------------------------------------------------
# 3. CARREGAR RUÍDOS EM MEMÓRIA
# ---------------------------------------------------------------------
print("\nCarregando ruídos...")
ruidos = [carregar_audio(c) for c in tqdm(arquivos_ruidos)]

# ---------------------------------------------------------------------
# 4. PROCESSAR: 1 PAR POR ÁUDIO LIMPO, DURAÇÃO COMPLETA
# ---------------------------------------------------------------------
print("\nGerando pares (limpo / ruidoso) preservando a duração completa...")

pares_salvos = 0
pares_descartados = 0

for caminho_limpo in tqdm(arquivos_limpos, desc="Misturando"):
    audio_limpo = carregar_audio(caminho_limpo)

    # descarta áudios muito silenciosos
    if rms(audio_limpo) < RMS_MIN_LIMIAR:
        pares_descartados += 1
        print(f"[AVISO] {os.path.basename(caminho_limpo)} descartado (RMS baixo).")
        continue

    # 🔀 escolhe ruído aleatório, trecho aleatório do tamanho do limpo, SNR aleatório
    ruido_escolhido = random.choice(ruidos)
    trecho_ruido   = extrair_trecho_aleatorio(ruido_escolhido, len(audio_limpo))
    snr            = random.uniform(SNR_MIN_DB, SNR_MAX_DB)

    limpo_n, ruidoso = misturar_com_snr(audio_limpo, trecho_ruido, snr)

    base_nome = os.path.splitext(os.path.basename(caminho_limpo))[0]
    nome_limpo   = f"{base_nome}_limpo.wav"
    nome_ruidoso = f"{base_nome}_ruidoso.wav"

    sf.write(os.path.join(DIR_SAIDA, nome_limpo),   limpo_n,  SR)
    sf.write(os.path.join(DIR_SAIDA, nome_ruidoso), ruidoso,  SR)
    pares_salvos += 1

print(f"\n✅ Pares salvos : {pares_salvos}")
print(f"⚠️  Descartados : {pares_descartados} (abaixo do limiar de RMS)")

# ---------------------------------------------------------------------
# 5. ZIPAR RESULTADO PARA DOWNLOAD
# ---------------------------------------------------------------------
shutil.make_archive("/content/audios_misturados", "zip", DIR_SAIDA)
print("\n📦 Arquivo ZIP gerado: /content/audios_misturados.zip")

try:
    from google.colab import files
    files.download("/content/audios_misturados.zip")
except ImportError:
    pass

print("\n🎧 Pronto! Os arquivos estão em:", DIR_SAIDA)

📤 Faça upload dos ÁUDIOS LIMPOS (motores, vozes, etc.):


Saving A-0221.wav to A-0221.wav
Saving A-0222.wav to A-0222.wav
Saving A-0223.wav to A-0223.wav
Saving A-0224.wav to A-0224.wav
Saving A-0225.wav to A-0225.wav
Saving A-0226.wav to A-0226.wav
Saving A-0227.wav to A-0227.wav
Saving A-0228.wav to A-0228.wav
Saving A-0229.wav to A-0229.wav
Saving A-0230.wav to A-0230.wav

📤 Faça upload dos RUÍDOS (ambientes, vento, etc.):


Saving E-1446.wav to E-1446.wav
Saving E-1522.wav to E-1522.wav
Saving E-1533.wav to E-1533.wav
Saving E-1568.wav to E-1568.wav
Saving E-1591.wav to E-1591.wav
Saving E-1611.wav to E-1611.wav
Saving E-1621.wav to E-1621.wav
Saving E-1627.wav to E-1627.wav
Saving E-1644.wav to E-1644.wav
Saving E-1647.wav to E-1647.wav

Procurando áudios...
  Áudios limpos: 10
  Ruídos       : 10

Carregando ruídos...


100%|██████████| 10/10 [00:22<00:00,  2.24s/it]



Gerando pares (limpo / ruidoso) preservando a duração completa...


Misturando: 100%|██████████| 10/10 [00:03<00:00,  2.52it/s]



✅ Pares salvos : 10
⚠️  Descartados : 0 (abaixo do limiar de RMS)

📦 Arquivo ZIP gerado: /content/audios_misturados.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎧 Pronto! Os arquivos estão em: /content/audios_misturados
